# Domain Adaptation: Importance Weighting - Hyperparameter Tuning

Goals:
- optimise the source-loss weight λ
- correct for distribution shift (larger weight for source compounds similar to the target)
- test all 3 properties

Design:
- Density ratio w(x) = p_target(x) / p_source(x)
  - estimated from the odds of a domain classifier (standardise → PCA 50-d → logistic regression, class_weight='balanced')
  - clipped to [0.1, 10] and normalised to mean 1
- model: shared encoder + target head + source head
  - target head: trained on target (in vivo) labels, used for prediction and evaluation
  - source head: trained with a w-weighted loss on source (in vitro) labels (only the encoder representation is shared)
- Total Loss = L_target + λ * mean(w_i * L_source_i)


In [1]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import spearmanr, gaussian_kde
from sklearn.metrics import r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import json
import warnings
warnings.filterwarnings('ignore')

ROOT = r'..'
DATA_DIR = os.path.join(ROOT, 'data', 'cv_datasets')
FEAT_DIR = os.path.join(ROOT, 'features')
RES_DIR = os.path.join(ROOT, 'results', 'importance_weighting_tuning')
os.makedirs(RES_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PROPERTIES = ['clearance', 'half_life', 'fu']
N_FOLDS = 5

# Hyperparameter Grid
IW_WEIGHTS = [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]

print("="*100)
print("Domain Adaptation: Importance Weighting - Hyperparameter Tuning")
print("="*100)
print(f"Device: {DEVICE}")
print(f"Properties: {PROPERTIES}")
print(f"N Folds: {N_FOLDS}")
print(f"Importance Weighting Strengths to test: {IW_WEIGHTS}\n")

# Shared training utilities: inner train/val split + val-based early stopping (test used only for final evaluation)
sys.path.insert(0, os.path.join(ROOT, '01_model'))
from train_utils import split_train_val, fit_early_stopping, target_smiles


Domain Adaptation: Importance Weighting - Hyperparameter Tuning
Device: cpu
Properties: ['clearance', 'half_life', 'fu']
N Folds: 5
Importance Weighting Strengths to test: [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]



In [2]:
print("Loading Graphormer embeddings...\n")

emb_path = os.path.join(FEAT_DIR, 'graphormer_cv_embeddings.npz')
emb_data = np.load(emb_path, allow_pickle=True)
emb_keys = emb_data['keys']
emb_X = emb_data['X']

smiles_to_emb = {smi: emb_X[i] for i, smi in enumerate(emb_keys)}

print(f"✓ Loaded {len(smiles_to_emb)} embeddings")
print(f"  Dim: {emb_X.shape[1]}\n")

Loading Graphormer embeddings...

✓ Loaded 58304 embeddings
  Dim: 768



In [3]:
class SharedEncoder(nn.Module):
    def __init__(self, d_in=768, d_hidden=256, dropout=0.2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, d_hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.d_out = d_hidden // 2

    def forward(self, x):
        return self.encoder(x)

class TaskHead(nn.Module):
    def __init__(self, d_in, d_hidden=128):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.ReLU(),
            nn.Linear(d_hidden, 1),
        )

    def forward(self, x):
        return self.head(x).squeeze(-1)

class ImportanceWeightingModel(nn.Module):
    def __init__(self, d_in=768):
        super().__init__()
        self.encoder = SharedEncoder(d_in)
        self.task_head = TaskHead(self.encoder.d_out)    # target (in vivo)
        self.source_head = TaskHead(self.encoder.d_out)  # source (in vitro), auxiliary for training

    def forward(self, x):
        features = self.encoder(x)
        output = self.task_head(features)
        return output, features

    def forward_source(self, x):
        return self.source_head(self.encoder(x))

print("✓ Model defined")

✓ Model defined


In [4]:
def get_smiles_col(df):
    if 'smiles' in df.columns:
        return 'smiles'
    elif 'Drug' in df.columns:
        return 'Drug'
    else:
        return None

def get_value_col(df):
    if 'value' in df.columns:
        return 'value'
    elif 'Y' in df.columns:
        return 'Y'
    else:
        return None

def load_source(prop, fold_idx):
    fold_dir = os.path.join(DATA_DIR, f'fold_{fold_idx}')
    csv_path = os.path.join(fold_dir, f'{prop}_source.csv')
    
    if not os.path.exists(csv_path):
        return None, None
    
    df = pd.read_csv(csv_path, low_memory=False)
    smi_col = get_smiles_col(df)
    val_col = get_value_col(df)
    
    if smi_col is None or val_col is None:
        return None, None
    
    smiles = df[smi_col].astype(str).tolist()
    
    X_list = []
    y_list = []
    for smi, y in zip(smiles, df[val_col].values):
        if smi in smiles_to_emb:
            X_list.append(smiles_to_emb[smi])
            y_list.append(y)
    
    X = np.stack(X_list).astype('float32') if X_list else np.array([])
    y = np.array(y_list, dtype='float32') if y_list else np.array([])
    return X, y

def load_target(prop, fold_idx, split):
    fold_dir = os.path.join(DATA_DIR, f'fold_{fold_idx}')
    csv_path = os.path.join(fold_dir, f'{prop}_target_{split}.csv')
    
    if not os.path.exists(csv_path):
        return None, None
    
    df = pd.read_csv(csv_path, low_memory=False)
    smi_col = get_smiles_col(df)
    val_col = get_value_col(df)
    
    if smi_col is None or val_col is None:
        return None, None
    
    smiles = df[smi_col].astype(str).tolist()
    
    X_list = []
    y_list = []
    for smi, y in zip(smiles, df[val_col].values):
        if smi in smiles_to_emb:
            X_list.append(smiles_to_emb[smi])
            y_list.append(y)
    
    X = np.stack(X_list).astype('float32') if X_list else np.array([])
    y = np.array(y_list, dtype='float32') if y_list else np.array([])
    return X, y

print("✓ Data loaders defined")

✓ Data loaders defined


In [5]:
_iw_cache = {}

def compute_importance_weights(X_src, X_tgt, n_components=50, clip=(0.1, 10.0), seed=0):
    """Density ratio w(x) = p_target(x) / p_source(x) via probabilistic domain classifier

    When trained with class_weight='balanced', the odds p(t|x)/p(s|x) are directly the density-ratio estimate.
    """
    X = np.vstack([X_src, X_tgt])
    d = np.r_[np.zeros(len(X_src)), np.ones(len(X_tgt))]
    Z = StandardScaler().fit_transform(X)
    Z = PCA(n_components=n_components, random_state=seed).fit_transform(Z)
    clf = LogisticRegression(C=0.1, max_iter=2000, class_weight='balanced')
    clf.fit(Z, d)
    p_t = clf.predict_proba(Z[:len(X_src)])[:, 1]
    w = p_t / np.clip(1.0 - p_t, 1e-6, None)
    w = np.clip(w, *clip)
    w = w / w.mean()
    ess = w.sum() ** 2 / (w ** 2).sum()
    print(f"    [IW] weights: min={w.min():.3f} median={np.median(w):.3f} max={w.max():.3f} | ESS={ess:.0f}/{len(w)}")
    return w.astype('float32')

def train_epoch_iw(model, X_src, y_src, weights, X_tgt, y_tgt, optimizer, criterion, iw_w=0.1, device='cpu'):
    """L_target + λ * mean(w * L_source): epochs count target batches, the source is randomly sampled each batch"""
    model.train()

    n = len(X_tgt)
    perm = np.random.permutation(n)

    loss_sum = 0.0
    n_batches = 0

    for i in range(0, n, 32):
        idx = perm[i:i + 32]
        idx_s = np.random.randint(0, len(X_src), len(idx))

        x_tgt = torch.tensor(X_tgt[idx], device=device, dtype=torch.float32)
        y_tgt_batch = torch.tensor(y_tgt[idx], device=device, dtype=torch.float32)
        x_src = torch.tensor(X_src[idx_s], device=device, dtype=torch.float32)
        y_src_batch = torch.tensor(y_src[idx_s], device=device, dtype=torch.float32)
        w_batch = torch.tensor(weights[idx_s], device=device, dtype=torch.float32)

        optimizer.zero_grad()

        y_pred_tgt, _ = model(x_tgt)
        tgt_loss = criterion(y_pred_tgt, y_tgt_batch).mean()

        y_pred_src = model.forward_source(x_src)
        src_loss = (criterion(y_pred_src, y_src_batch) * w_batch).mean()

        loss = tgt_loss + iw_w * src_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        loss_sum += loss.item()
        n_batches += 1

    return loss_sum / max(n_batches, 1)

@torch.no_grad()
def evaluate(model, X, y, device):
    model.eval()
    X = np.asarray(X, dtype='float32')
    y = np.asarray(y, dtype='float32')
    
    x = torch.tensor(X, device=device, dtype=torch.float32)
    y_pred, _ = model(x)
    y_pred = y_pred.cpu().numpy()
    
    y_pred = np.nan_to_num(y_pred, nan=0.0, posinf=1e6, neginf=-1e6)
    
    r = np.corrcoef(y, y_pred)[0, 1]
    rho, _ = spearmanr(y, y_pred)
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(np.mean((y - y_pred) ** 2))
    mae = np.mean(np.abs(y - y_pred))
    mbe = np.mean(y_pred - y)
    
    return {'r': float(r), 'rho': float(rho), 'r2': float(r2), 'rmse': float(rmse), 'mae': float(mae), 'mbe': float(mbe)}

print("✓ Training functions defined")

✓ Training functions defined


In [6]:
print("\n" + "="*100)
print("HYPERPARAMETER TUNING: Importance Weighting Strength Grid Search")
print("="*100 + "\n")

tuning_results = {}

for prop in PROPERTIES:
    print(f"\n{'='*80}")
    print(f"Property: {prop.upper()}")
    print(f"{'='*80}")
    
    tuning_results[prop] = {}
    
    for iw_w in IW_WEIGHTS:
        print(f"\n  Testing Importance Weighting Strength = {iw_w}")
        fold_results = []
        
        for fold_idx in range(N_FOLDS):
            X_src, y_src = load_source(prop, fold_idx)
            X_tgt_train, y_tgt_train = load_target(prop, fold_idx, 'train')
            X_tgt_test, y_tgt_test = load_target(prop, fold_idx, 'test')
            
            if X_src is None or len(X_src) == 0:
                continue
            
            # The source is not truncated; everything is used (randomly sampled each batch during training)
            X_src = np.asarray(X_src, dtype='float32')
            y_src = np.asarray(y_src, dtype='float32')
            # target train → scaffold-based inner train/val (early stopping and λ selection on val, test used only for final evaluation)
            X_tgt_train, y_tgt_train, X_tgt_val, y_tgt_val = split_train_val(
                np.asarray(X_tgt_train, dtype='float32'), np.asarray(y_tgt_train, dtype='float32'), fold_idx,
                smiles=target_smiles(DATA_DIR, prop, fold_idx, 'train', smiles_to_emb))
            X_tgt_test = np.asarray(X_tgt_test, dtype='float32')
            y_tgt_test = np.asarray(y_tgt_test, dtype='float32')
            
            # Compute importance weights
            if (prop, fold_idx) not in _iw_cache:
                _iw_cache[(prop, fold_idx)] = compute_importance_weights(X_src, X_tgt_train)
            weights = _iw_cache[(prop, fold_idx)]
            
            
            torch.manual_seed(fold_idx)
            np.random.seed(fold_idx)
            
            model = ImportanceWeightingModel().to(DEVICE)
            optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
            criterion = nn.HuberLoss(delta=1.0, reduction='none')
            
            def run_epoch():
                train_epoch_iw(model, X_src, y_src, weights, X_tgt_train, y_tgt_train, optimizer, criterion, iw_w, DEVICE)
            
            best_epoch, _ = fit_early_stopping(
                model, run_epoch, lambda m, X, y: evaluate(m, X, y, DEVICE), X_tgt_val, y_tgt_val)
            val_metrics = evaluate(model, X_tgt_val, y_tgt_val, DEVICE)
            metrics = evaluate(model, X_tgt_test, y_tgt_test, DEVICE)
            metrics.update({f'val_{k}': v for k, v in val_metrics.items()})
            metrics['best_epoch'] = best_epoch
            fold_results.append(metrics)
        
        if fold_results:
            results_df = pd.DataFrame(fold_results)
            tuning_results[prop][iw_w] = {
                'rho_mean': float(results_df['rho'].mean()),
                'rho_std': float(results_df['rho'].std()),
                'r2_mean': float(results_df['r2'].mean()),
                'r2_std': float(results_df['r2'].std()),
                'rmse_mean': float(results_df['rmse'].mean()),
                'rmse_std': float(results_df['rmse'].std()),
                'val_rho_mean': float(results_df['val_rho'].mean()),
                'val_r2_mean': float(results_df['val_r2'].mean()),
            }
            
            print(f"    ρ: {results_df['rho'].mean():.4f}±{results_df['rho'].std():.4f} | R²: {results_df['r2'].mean():.4f}±{results_df['r2'].std():.4f}")
            results_df.to_csv(os.path.join(RES_DIR, f'{prop}_iw{iw_w}.csv'), index=False)


HYPERPARAMETER TUNING: Importance Weighting Strength Grid Search


Property: CLEARANCE

  Testing Importance Weighting Strength = 0.01
    [IW] weights: min=0.159 median=0.172 max=15.916 | ESS=1925/14386
    [IW] weights: min=0.160 median=0.179 max=15.969 | ESS=1933/14386
    [IW] weights: min=0.165 median=0.170 max=16.525 | ESS=1894/14386
    [IW] weights: min=0.168 median=0.168 max=16.754 | ESS=1783/14386
    [IW] weights: min=0.165 median=0.165 max=16.465 | ESS=1801/14386
    ρ: 0.3325±0.0592 | R²: 0.0795±0.0608

  Testing Importance Weighting Strength = 0.05
    ρ: 0.3544±0.0434 | R²: 0.0737±0.0607

  Testing Importance Weighting Strength = 0.1
    ρ: 0.3426±0.0461 | R²: 0.0647±0.0657

  Testing Importance Weighting Strength = 0.3
    ρ: 0.3463±0.0316 | R²: 0.0755±0.0324

  Testing Importance Weighting Strength = 0.5
    ρ: 0.3509±0.0719 | R²: 0.0811±0.0518

  Testing Importance Weighting Strength = 1.0
    ρ: 0.3456±0.0615 | R²: 0.0721±0.0479

Property: HALF_LIFE

  Testing Impor

In [7]:
print(f"\n\n{'='*100}")
print("IMPORTANCE WEIGHTING TUNING RESULTS SUMMARY")
print(f"{'='*100}\n")

for prop in PROPERTIES:
    print(f"\n{prop.upper()}:")
    print(f"  {'IW_W':<10} {'Spearman ρ':<20} {'R²':<20} {'RMSE':<20}")
    print(f"  {'-'*70}")
    
    best_rho = -1
    best_r2 = -1
    best_w_rho = None
    best_w_r2 = None
    
    for iw_w in IW_WEIGHTS:
        if iw_w in tuning_results[prop]:
            res = tuning_results[prop][iw_w]
            rho_str = f"{res['rho_mean']:.4f}±{res['rho_std']:.4f}"
            r2_str = f"{res['r2_mean']:.4f}±{res['r2_std']:.4f}"
            rmse_str = f"{res['rmse_mean']:.4f}±{res['rmse_std']:.4f}"
            
            marker = ""
            if res['val_rho_mean'] > best_rho:
                best_rho = res['val_rho_mean']
                best_w_rho = iw_w
                marker += "(Best val ρ)"
            if res['val_r2_mean'] > best_r2:
                best_r2 = res['val_r2_mean']
                best_w_r2 = iw_w
                marker += "(Best val R²)"
            
            print(f"  {iw_w:<10.2f} {rho_str:<20} {r2_str:<20} {rmse_str:<20} {marker}")
    
    print(f"\n  Best IW_W for ρ: {best_w_rho} ({best_rho:.4f})")
    print(f"  Best IW_W for R²: {best_w_r2} ({best_r2:.4f})")



IMPORTANCE WEIGHTING TUNING RESULTS SUMMARY


CLEARANCE:
  IW_W       Spearman ρ           R²                   RMSE                
  ----------------------------------------------------------------------
  0.01       0.3325±0.0592        0.0795±0.0608        0.9575±0.1075        (Best val ρ)(Best val R²)
  0.05       0.3544±0.0434        0.0737±0.0607        0.9599±0.1006        (Best val ρ)
  0.10       0.3426±0.0461        0.0647±0.0657        0.9640±0.0951        
  0.30       0.3463±0.0316        0.0755±0.0324        0.9588±0.0909        
  0.50       0.3509±0.0719        0.0811±0.0518        0.9559±0.0945        (Best val ρ)(Best val R²)
  1.00       0.3456±0.0615        0.0721±0.0479        0.9604±0.0921        (Best val R²)

  Best IW_W for ρ: 0.5 (0.4002)
  Best IW_W for R²: 1.0 (0.1675)

HALF_LIFE:
  IW_W       Spearman ρ           R²                   RMSE                
  ----------------------------------------------------------------------
  0.01       0.3801±0.0689  

In [8]:
tuning_summary = {}
for prop in PROPERTIES:
    tuning_summary[prop] = {}
    for iw_w, results in tuning_results[prop].items():
        tuning_summary[prop][str(iw_w)] = {
            'rho_mean': float(results['rho_mean']),
            'rho_std': float(results['rho_std']),
            'r2_mean': float(results['r2_mean']),
            'r2_std': float(results['r2_std']),
            'rmse_mean': float(results['rmse_mean']),
            'rmse_std': float(results['rmse_std']),
            'val_rho_mean': float(results['val_rho_mean']),
            'val_r2_mean': float(results['val_r2_mean']),
        }

with open(os.path.join(RES_DIR, 'tuning_results.json'), 'w') as f:
    json.dump(tuning_summary, f, indent=2)

print(f"\n\n✓ Importance Weighting tuning results saved to {RES_DIR}")
print("\nImportance Weighting Hyperparameter Tuning Complete!")



✓ Importance Weighting tuning results saved to ..\results\importance_weighting_tuning

Importance Weighting Hyperparameter Tuning Complete!
